# Lab: Application 

Detect water bodies using the Otsu threshold on the NDWI index from a satellite image acquired by Sentinel-2.

This application takes as input Copernicus Sentinel-2 data and detects water bodies by applying the Otsu thresholding technique on the Normalized Difference Water Index (NDWI).

Link to documentation: https://eoap.github.io/quickwin/python-env/water_bodies/

## Application code



The application requires a few libraries:

In [ ]:
# Use the module-owned processing implementation from this checkout.
import inspect
import sys
import numpy as np
from pathlib import Path

package = Path("../water-bodies/command-line-tools/detect-water-body").resolve()
sys.path.insert(0, str(package))
from detect_water_body.main import (
    aoi2box, crop, get_asset, main, normalized_difference, threshold
)
import pystac


Let's ignore the numpy warnings about divisions by zero:

In [ ]:

np.seterr(divide="ignore", invalid="ignore")


Let's define a crop function taking as argument:

- a pystac `Asset` object
- a bounding box
- the EPSG code of the bounding box coordinates

This function returns a numpy array.

In [ ]:
# Inspect the implementation used by the processing container.
print(inspect.getsource(crop))


Let's define a function to apply the normalized differences between two numpy arrays and produce a numpy array:

In [ ]:
# Inspect the implementation used by the processing container.
print(inspect.getsource(normalized_difference))


Let's define a function to apply the Ostu automatic threshold taking as input a numpy array and producing a numpy array:

In [ ]:
# Inspect the implementation used by the processing container.
print(inspect.getsource(threshold))


Let's define a helper function to split a bounding box expressed as `"min_x,min_y,max_x, max_y"` and return a list of floats:

In [ ]:
# Inspect the implementation used by the processing container.
print(inspect.getsource(aoi2box))


Let's define a function that takes a STAC Item and a common band name as input and returns the STAC Asset whose common band property matches the input value

In [ ]:
# Inspect the implementation used by the processing container.
print(inspect.getsource(get_asset))


Let's define a `main` function and use `click` to decorate it producing a command line tool interface.\

The `main` function reads a STAC item, and area of interest, the EPSG code used to express the bounding box coordinates and the list of common bands names for the normalized difference:

In [ ]:
# Inspect the actual Click command callback.
print(inspect.getsource(main.callback))


The `main` function is a command line tool that takes a few input parameters. 

The help can be printed with:

In [ ]:
from click.testing import CliRunner

runner = CliRunner()
result = runner.invoke(main, ['--help'])

print(result.output)

This command line tool can be invoked with:

In [ ]:
arguments = ["--input-item", "https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A",
            "--aoi", "-121.399,39.834,-120.74,40.472",
            "--epsg", "EPSG:4326",
            "--band", "green",
            "--band", "nir"]

runner = CliRunner()
result = runner.invoke(main, args=arguments)

print(result.output)
if result.exception:
    raise result.exception
assert result.exit_code == 0, result.output


The application generated a STAC Catalog that can be read:

In [ ]:
cat = pystac.read_file("catalog.json")

cat.describe()

In [ ]:
item = next(cat.get_items())

item